# M1.5: Visualising energy data

## Objectives

At the end of this notebook you should be able to:

- follow the conventions that make energy plots trustworthy: units on every axis, local time for people, one colour per carrier, honest baselines
- show patterns across hours, days and seasons: weekly time series, average daily profiles, heatmaps
- build **duration curves** for load and residual load, and read what they mean for capacity and flexibility
- draw a **stacked dispatch** chart with prices, and an annual energy mix
- use plots as a **check**: a figure that looks wrong usually is wrong

Your visualisation skills (Tableau, dashboards) carry over; what is new is *which* plots energy analysts rely on and what each one reveals. The data is one **synthetic** year built from M1.2–M1.4: load shaped like Croatia's, wind from the M1.4 power curve, solar from the M1.4 clear-sky model with random clouds, dispatched in your M1.2 network. Plausible in shape, but not measured — every number below is illustrative.

In [1]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pvlib
import pypsa
from scipy import stats

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (10, 3.5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

### The data: one synthetic year

The helper functions below build the year and dispatch it. Note the **colour dictionary**: one fixed colour per carrier, used in every plot. Readers learn "orange = gas" once; changing colours between figures forces them to re-read every legend.

In [2]:
CARRIER_COLORS = {"nuclear": "#7B4F9E", "hydro": "#2A7AB8", "wind": "#4FB0C6", "solar": "#F2C14E",
                  "coal": "#5C4033", "gas": "#E07B39", "imports": "#9E9E9E", "load": "#000000"}


def power_curve(v, v_cut_in=3.0, v_rated=12.0, v_cut_out=25.0):
    """Generic per-unit turbine output (M1.4)."""
    v = np.asarray(v, dtype=float)
    rising = (v ** 3 - v_cut_in ** 3) / (v_rated ** 3 - v_cut_in ** 3)
    out = np.where((v >= v_cut_in) & (v < v_rated), rising, 0.0)
    return np.where((v >= v_rated) & (v <= v_cut_out), 1.0, out)


def synthetic_year(year: int = 2025, seed: int = 3) -> pd.DataFrame:
    """SYNTHETIC hourly year (UTC, interval start): load in MW, wind and solar availability in p.u.

    Built from the physics of M1.4 (power curve, clear-sky sun) plus random weather.
    Plausible in shape for a Croatia-like system, but NOT measured data.
    """
    rng = np.random.default_rng(seed)
    idx = pd.date_range(f"{year}-01-01", f"{year + 1}-01-01", freq="h", tz="UTC", inclusive="left")
    local = idx.tz_convert("Europe/Zagreb")
    doy, hour = local.dayofyear.to_numpy(), local.hour.to_numpy()
    weekend = local.dayofweek.to_numpy() >= 5
    load = (2100 + 250 * np.cos(2 * np.pi * (doy - 15) / 365)          # winter peak
            + 350 * np.sin((hour - 7) / 24 * 2 * np.pi) - 200 * weekend
            + rng.normal(0, 40, len(idx)))
    z = np.empty(len(idx)); z[0] = 0.0                                  # persistent weather
    for t in range(1, len(idx)):
        z[t] = 0.97 * z[t - 1] + np.sqrt(1 - 0.97 ** 2) * rng.normal()
    season = 1 + 0.15 * np.cos(2 * np.pi * (doy - 15) / 365)            # windier winters
    speeds = stats.weibull_min.ppf(stats.norm.cdf(z), 2.0, scale=7.5) * season
    clouds = np.empty(len(idx)); clouds[0] = 0.8
    for t in range(1, len(idx)):
        clouds[t] = np.clip(0.95 * clouds[t - 1] + 0.05 * 0.75 + rng.normal(0, 0.05), 0.15, 1.0)
    zen = pvlib.solarposition.get_solarposition(idx + pd.Timedelta("30min"), 43.51, 16.44)["apparent_zenith"]
    ghi_clear = pvlib.clearsky.haurwitz(zen)["ghi"].fillna(0).to_numpy()
    return pd.DataFrame({"load_MW": load.round(1), "wind_pu": power_curve(speeds).round(3),
                         "solar_pu": np.clip(ghi_clear / 1000 * 0.86 * clouds, 0, 1).round(3)}, index=idx)


def build_and_solve(profiles: pd.DataFrame) -> pypsa.Network:
    """M1.2's stylised single-bus system, dispatched for the given profiles (naive UTC snapshots)."""
    n = pypsa.Network()
    n.set_snapshots(profiles.index.tz_localize(None))
    n.add("Carrier", "AC")
    n.add("Bus", "HR", carrier="AC")
    for name, co2 in [("nuclear", 0.0), ("hydro", 0.0), ("wind", 0.0), ("solar", 0.0),
                      ("coal", 0.34), ("gas", 0.20), ("imports", 0.0)]:
        n.add("Carrier", name, co2_emissions=co2)
    n.add("Load", "demand", bus="HR", carrier="AC", p_set=profiles["load_MW"].to_numpy())
    for name, carrier, p_nom, mc, eff in [("nuclear_share", "nuclear", 350, 10, 0.33),
                                          ("hydro_ror", "hydro", 1500, 2, 1.0),
                                          ("wind", "wind", 1000, 0, 1.0), ("solar", "solar", 800, 0, 1.0),
                                          ("coal", "coal", 210, 95, 0.38), ("gas_ccgt", "gas", 700, 110, 0.58),
                                          ("gas_ocgt", "gas", 300, 160, 0.38),
                                          ("imports", "imports", 1500, 120, 1.0)]:
        n.add("Generator", name, bus="HR", carrier=carrier, p_nom=p_nom, marginal_cost=mc, efficiency=eff)
    n.generators_t.p_max_pu["hydro_ror"] = 0.45
    n.generators_t.p_max_pu["wind"] = profiles["wind_pu"].to_numpy()
    n.generators_t.p_max_pu["solar"] = profiles["solar_pu"].to_numpy()
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n


profiles = synthetic_year()
n = build_and_solve(profiles)
local_index = profiles.index.tz_convert("Europe/Zagreb")
print(f"load {profiles.load_MW.sum() / 1e6:.1f} TWh, wind CF {profiles.wind_pu.mean():.2f}, "
      f"solar CF {profiles.solar_pu.mean():.2f}, total cost {n.objective / 1e6:.0f} M EUR")

load 17.9 TWh, wind CF 0.25, solar CF 0.15, total cost 657 M EUR


### Rule 1: units, time zone and source on every plot

Every axis gets a unit (MW, MWh, €/MWh, p.u.). Time axes for people use **local time** — data stay in UTC (M1.1) and are converted only for display. And synthetic or modelled data say so in the title. A one-week view of the inputs:

In [3]:
week = slice("2025-01-13", "2025-01-19")
w = profiles.set_index(local_index).loc[week]
fig, ax = plt.subplots()
ax.plot(w.index, w.load_MW, color=CARRIER_COLORS["load"], lw=1.2, label="load")
ax.plot(w.index, w.wind_pu * 1000, color=CARRIER_COLORS["wind"], label="wind available (1,000 MW)")
ax.plot(w.index, w.solar_pu * 800, color=CARRIER_COLORS["solar"], label="solar available (800 MW)")
ax.set_ylabel("MW"); ax.set_xlabel("local time (Europe/Zagreb)")
ax.set_title("A winter week — synthetic data"); ax.legend(ncol=3, fontsize=8)
plt.show()

Load follows a strong daily rhythm; solar is a short midday bump in winter; wind changes over days, not hours, because weather systems are persistent. Those three time scales are the core challenge of a renewable power system.

### Average daily profiles by season

Grouping by **local** hour and season compresses a year into one readable figure. (Group by UTC hour and every summer curve shifts by an hour — DST from M1.1.)

In [4]:
season_of_month = {12: "winter", 1: "winter", 2: "winter", 3: "spring", 4: "spring", 5: "spring",
                   6: "summer", 7: "summer", 8: "summer", 9: "autumn", 10: "autumn", 11: "autumn"}
seasons = local_index.month.map(season_of_month)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharex=True)
for col, ax, unit in [("load_MW", axes[0], "MW"), ("solar_pu", axes[1], "p.u.")]:
    profile = profiles[col].groupby([seasons, local_index.hour]).mean().unstack(0)
    profile[["winter", "spring", "summer", "autumn"]].plot(ax=ax)
    ax.set_xlabel("local hour"); ax.set_ylabel(unit); ax.set_title(f"average daily {col.split('_')[0]}")
plt.tight_layout(); plt.show()

Winter load is higher at every hour, and summer solar at noon is about twice winter solar. On one page the seasonal mismatch is obvious: demand is highest when solar is weakest. (The synthetic load peaks in the early afternoon — a simplification. Real Croatian load also has a pronounced evening peak; plot your ENTSO-E data from M1.1 the same way and compare.)

### Heatmaps: a whole year in one picture

A **heatmap** with days on one axis and hours on the other shows all 8,760 values at once — patterns, gaps and outliers jump out. In local time, the spring DST day has a missing hour and the autumn day a doubled one; averaging the duplicate keeps one value per cell.

In [5]:
def year_matrix(series: pd.Series) -> pd.DataFrame:
    """Day x local-hour matrix of an hourly UTC series (DST: missing hour NaN, doubled hour averaged)."""
    s = series.copy()
    s.index = s.index.tz_convert("Europe/Zagreb")
    return s.groupby([s.index.date, s.index.hour]).mean().unstack()

solar_matrix = year_matrix(profiles.solar_pu)
local_year = [d for d in solar_matrix.index if d.year == 2025]      # drop the spill-over into 2026
solar_matrix = solar_matrix.loc[local_year]
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.grid(False)
im = ax.imshow(solar_matrix.T, aspect="auto", origin="lower", cmap="viridis", interpolation="none")
ax.set_xlabel("day of year"); ax.set_ylabel("local hour"); ax.set_title("solar availability — synthetic")
fig.colorbar(im, ax=ax, label="p.u.")
plt.show()
empty = solar_matrix.stack(future_stack=True)
print(f"matrix shape {solar_matrix.shape}; empty cells: {list(empty[empty.isna()].index)}")

matrix shape (365, 24); empty cells: [(datetime.date(2025, 1, 1), 0), (datetime.date(2025, 3, 30), 2)]


The bright band widens in summer (longer days) and jumps by an hour at both DST changes — a clean check that the time handling is right. Random dark stripes are cloudy days.

Exactly two cells are empty, and both are time-zone effects worth understanding: **local hour 2 on 30 March** does not exist (clocks jump from 02:00 to 03:00), and **local midnight on 1 January** lies before the data start, because the year was built in UTC and Croatia is one hour ahead in winter. (The last UTC hour of the year spills into 1 January 2026 local time — that row was dropped above.) In real data, heatmaps are also the fastest way to spot gaps and stuck meters from M1.1.

### Duration curves: load and residual load

A **load duration curve** sorts the hours from highest to lowest load (M1.1 exercise). The **residual load** — load minus wind and solar — is what the *rest* of the system must cover. Plot both:

In [6]:
wind_MW = n.generators_t.p_max_pu["wind"].to_numpy() * 1000
solar_MW = n.generators_t.p_max_pu["solar"].to_numpy() * 800
residual = profiles.load_MW.to_numpy() - wind_MW - solar_MW
hours = np.arange(1, len(residual) + 1)

fig, ax = plt.subplots()
ax.plot(hours, np.sort(profiles.load_MW.to_numpy())[::-1], color="black", label="load")
ax.plot(hours, np.sort(residual)[::-1], color=CARRIER_COLORS["gas"], label="residual load (load − wind − solar)")
ax.set_xlabel("hours per year at or above this level"); ax.set_ylabel("MW"); ax.set_ylim(bottom=0)
ax.set_title("duration curves — synthetic year"); ax.legend()
plt.show()

energy_gap_TWh = (profiles.load_MW.sum() - residual.sum()) / 1e6
assert np.isclose(energy_gap_TWh, (wind_MW.sum() + solar_MW.sum()) / 1e6)
print(f"peak load {profiles.load_MW.max():.0f} MW vs peak residual {residual.max():.0f} MW")
print(f"area between the curves = wind + solar energy = {energy_gap_TWh:.2f} TWh")

peak load 2803 MW vs peak residual 2668 MW
area between the curves = wind + solar energy = 3.24 TWh


Renewables shift the curve down by a lot of **energy** (the area between the curves, 3.2 TWh), but the **peak** residual load is almost as high as the peak load: there are windless winter evenings. Capacity to cover those hours is still needed — it just runs far fewer hours. This one plot explains much of the debate about firm capacity and flexibility (Module 05).

The y-axis starts at 0 on purpose: a truncated axis would make the difference between the curves look far larger than it is.

### Stacked dispatch with prices

The workhorse plot of every energy model: stacked generation by carrier, load on top, and the price on a second axis. Stack in merit order (cheapest at the bottom), so the marginal technology sits at the top of the stack.

In [7]:
by_carrier = n.generators_t.p.T.groupby(n.generators.carrier).sum().T
order = ["nuclear", "hydro", "wind", "solar", "coal", "gas", "imports"]
d = by_carrier[order].set_index(local_index).loc["2025-07-07":"2025-07-13"]
load_w = pd.Series(n.loads_t.p_set["demand"].to_numpy(), index=local_index).loc[d.index]
price_w = pd.Series(n.buses_t.marginal_price["HR"].to_numpy(), index=local_index).loc[d.index]

fig, ax = plt.subplots(figsize=(10, 4))
ax.stackplot(d.index, d.T.to_numpy(), labels=order, colors=[CARRIER_COLORS[c] for c in order])
ax.plot(d.index, load_w, color="black", lw=1.2, label="load")
ax.set_ylabel("MW"); ax.set_xlabel("local time")
ax2 = ax.twinx(); ax2.step(d.index, price_w, color="crimson", lw=1, where="post", label="price")
ax2.set_ylabel("EUR/MWh", color="crimson"); ax2.set_ylim(0, 130)
ax.set_title("dispatch and price, a summer week — synthetic data")
ax.legend(ncol=4, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.14))
plt.show()
assert np.allclose(d.sum(axis=1), load_w), "the stack must add up to the load"


The assert checks what the eye cannot: the stack exactly matches the load in every hour. Now read the picture. On weekdays the CCGT is marginal through most of the day (110 €/MWh) and coal at night (95 €/MWh); imports set the price in a few peak hours. At the weekend, strong wind pushes the price down to the marginal cost of nuclear and hydro (10 and 2 €/MWh).

Look closely at those windy hours: the model turns the **nuclear plant down to zero**. A real nuclear plant cannot do that — it has a minimum stable output and needs many hours to restart. The model is missing a constraint (minimum load and start-up costs: unit commitment, Module 03), and the plot makes that visible in seconds. This is why you plot model results: **a figure that looks wrong usually is wrong.**

A price drawn as a **step** (`where="post"`) is honest: the price is constant within each hour, not a straight line between them.

### Price versus residual load: the merit order, visible

One scatter plot shows why prices are what they are: each dot is an hour.

In [8]:
price = n.buses_t.marginal_price["HR"].to_numpy()
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(residual, price, s=4, alpha=0.3, color="crimson")
ax.set_xlabel("residual load (MW)"); ax.set_ylabel("price (EUR/MWh)")
ax.set_title("price vs residual load — synthetic year")
plt.show()
rho = stats.spearmanr(residual, price).statistic
print(f"rank correlation price vs residual load: {rho:.2f}")

rank correlation price vs residual load: 0.85


The dots fall on **steps**: each plateau is one technology's marginal cost (2 hydro, 10 nuclear, 95 coal, 110 CCGT, 120 imports). The steps switch at sharp thresholds — 675, 1,025, 1,235 and 1,935 MW — which are exactly the cumulative available capacities of hydro (1,500 MW × 0.45), + nuclear (350), + coal (210), + CCGT (700). In this model every plant except wind and solar has constant availability, so residual load alone determines the price. The rank correlation of 0.85 (not 1, because many hours share the same price) says the same in one number.

Real prices blur these steps: fuel and CO₂ prices change, plants have outages, neighbours trade, and bidders add mark-ups. You will see that in real Croatian and European prices in Module 02 — and this plot is the tool to see it.

### Annual energy mix

In [9]:
monthly_TWh = by_carrier[order].set_index(local_index).groupby(local_index.month).sum() / 1e6
ax = monthly_TWh.plot.bar(stacked=True, color=[CARRIER_COLORS[c] for c in order], figsize=(10, 3.5),
                          width=0.8, rot=0)
ax.set_xlabel("month"); ax.set_ylabel("TWh"); ax.set_title("monthly generation mix — synthetic year")
ax.legend(ncol=7, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.15))
plt.show()
assert np.isclose(monthly_TWh.to_numpy().sum(), profiles.load_MW.sum() / 1e6)

Solar's share grows in summer; winter needs more coal, gas and imports. The assert checks that the bars add up to annual demand — a mismatch here would mean a lost carrier or a unit error.

## Check your understanding

**Reading plots**

1. In the duration curves, what does the residual load at the far right (the lowest hours) tell you? What would happen if it became negative?
2. Why is the price-versus-residual-load plot made of horizontal bands rather than a smooth line?

**Making plots**

1. Redraw the daily profiles grouping by **UTC** hour instead of local hour. What changes in the summer curves, and why?
2. Change the dispatch plot to a winter week. Which carrier is at the top of the stack most of the time, and what does that say about the price?

**Thinking ahead**

The residual load curve is the basis of the **screening curve** method for choosing which power plants to build (Module 05). Which part of the curve would you serve with plants that are expensive to build but cheap to run, and which with plants that are cheap to build but expensive to run?